# Evaluate — Full Pipeline Comparison

Compares all inference pipelines against CVAT ground truth(GT) annotations.

**Input:**
- One or more crop runs from `infer_cropbased.ipynb` (`crop_results/{RUN_NAME}/`)
- Optionally one or more YOLO runs from `infer_yolo.ipynb` (`yolo_results/{RUN_NAME}/`)
- GT annotations from CVAT YOLO 1.1 export (`e2e_yolo_annotations/{camera}/obj_train_data/`)

**Matching method:** center-point — a prediction matches GT if either center point falls
inside the other's bounding box. This is more lenient than IoU and appropriate for small insects.

**Output:**
```
evaluation/
  summary.json      ← P/R/F1/TP/FP/FN per pipeline
  eval_results.csv  ← per-crop match/miss table (tp/fp/fn/rejected_as_bg)
  pr_curves.png     ← precision-recall curves for all pipelines
```

## Cell 1 — Environment

Set your local path. Only edit `BASE_DIR`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Images + annotations from local /content/ (extracted from zip)
    BASE_DIR = Path('/content/pollinator-colab')
    # Results from Drive (saved by infer_cropbased)
    DRIVE_BASE = Path('/content/drive/MyDrive/pollinator-colab')
else:
    BASE_DIR = Path('/Users/lianshi/Downloads/bachelor thesis'
                    '/automated-ecological-image-analysis'
                    '/ml-pipelines/notebooks/pollinator-classification')

IMAGE_ROOT        = BASE_DIR / 'Insects_images' / 'e2e_evaluation_images'
GT_ANN_ROOT       = BASE_DIR / 'Insects_images' / 'e2e_yolo_annotations'
MODEL_DIR         = BASE_DIR / 'models'
# On Colab: read results from Drive so evaluate works even in a new session
CROP_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'Insects_images' / 'crop_results'
YOLO_RESULTS_ROOT = (DRIVE_BASE if IN_COLAB else BASE_DIR) / 'Insects_images' / 'yolo_results'
INSECTNET_W       = BASE_DIR / 'InsectNet' / 'model.pth'
LABELED_DIR       = BASE_DIR / 'Insects_images' / 'annotated_crops'

CROP_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Env              : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR         : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'IMAGE_ROOT       : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'MODEL_DIR        : {MODEL_DIR}  exists={MODEL_DIR.exists()}')


## Cell 2 — Which runs to evaluate  ← **edit this**

List the runs you want to compare:

```python
CROP_RUNS = {
    'run_01_lm_on':  CROP_RESULTS_ROOT / 'run_01_lm_on',
    'run_02_lm_off': CROP_RESULTS_ROOT / 'run_02_lm_off',
}
YOLO_RUNS = {
    'yolo_v1': YOLO_RESULTS_ROOT / 'yolo_run_01',
}
```

Each crop run can contain multiple pipelines (e.g. `two_stage`, `five_class_eff`, `five_class_ins`).
The evaluate notebook auto-detects which pipelines are present from the CSV column names.
All pipelines from all runs appear in the final comparison table.

In [ ]:
# ── Crop runs ────────────────────────────────────────────────────
# Each key is a display name, value is the run folder
CROP_RUNS = {
    'run_01': CROP_RESULTS_ROOT / 'run_01',
    # 'run_02_lm_off': CROP_RESULTS_ROOT / 'run_02_lm_off',  # uncomment to compare
}

# ── YOLO runs ─────────────────────────────────────────────────────
YOLO_RUNS = {
    # 'yolo_run_01': YOLO_RESULTS_ROOT / 'yolo_run_01',
}

# ── GT settings ──────────────────────────────────────────────────
GT_CLASSES   = ['bumblebee', 'fly', 'butterfly', 'other']
STRIP_HEIGHT = 120

# ── YOLO inference settings (only if YOLO_RUNS is not empty) ─────
YOLO_WEIGHTS = MODEL_DIR / 'yolo_best.pt'
YOLO_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other']
SAHI_SLICE   = 640; SAHI_OVERLAP = 0.2; SAHI_CONF = 0.20; NMS_IOU = 0.45

EVAL_DIR = BASE_DIR / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('Runs to evaluate:')
for name, path in {**CROP_RUNS, **YOLO_RUNS}.items():
    print(f'  {name}: {path}  exists={path.exists()}')


## Cell 3 — Imports

Standard libraries. Just run.

In [ ]:
import csv, json as _json
from pathlib import Path
from collections import defaultdict
import numpy as np, cv2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt


## Cell 4 — Load ground truth

Reads CVAT YOLO 1.1 annotations from `e2e_yolo_annotations/`.

**Expected structure:**
```
e2e_yolo_annotations/{camera}/
  obj_train_data/
    WSCT0001.txt   ← YOLO format: class cx cy w h (normalised)
    WSCT0002.txt
  obj.names        ← class names in order (e.g. bumblebee, fly, butterfly, other)
```

The folder name in `e2e_yolo_annotations/` must match the folder name in `e2e_evaluation_images/`.
Prints a count per class so you can verify the annotations loaded correctly.

In [ ]:
def load_gt(gt_ann_root, image_root, gt_classes, strip_height=120):
    print('Loading ground truth...')
    gt = {}
    for cam_dir in sorted(Path(gt_ann_root).iterdir()):
        if not cam_dir.is_dir(): continue
        lbl_dir = cam_dir/'obj_train_data'
        img_dir = Path(image_root)/cam_dir.name
        if not lbl_dir.exists() or not img_dir.exists(): continue
        names_f = cam_dir/'obj.names'
        cls_names = ([l.strip() for l in names_f.read_text().splitlines() if l.strip()]
                     if names_f.exists() else gt_classes)
        n_cam = 0
        for txt in sorted(lbl_dir.glob('*.txt')):
            img_p = None
            for ext in ('.JPG','.jpg','.jpeg','.png'):
                cand = img_dir/(txt.stem+ext)
                if cand.exists(): img_p=cand; break
            if img_p is None: continue
            img = cv2.imread(str(img_p))
            if img is None: continue
            H,W = img.shape[:2]
            if strip_height>0 and H>strip_height: H-=strip_height
            boxes = []
            for line in txt.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts)<5: continue
                try: ci,cx,cy,bw,bh=int(parts[0]),*[float(x) for x in parts[1:5]]
                except ValueError: continue
                cname = cls_names[ci] if ci<len(cls_names) else str(ci)
                boxes.append({'cls':cname,
                              'x1':(cx-bw/2)*W,'y1':(cy-bh/2)*H,
                              'x2':(cx+bw/2)*W,'y2':(cy+bh/2)*H})
            gt[str(img_p)] = boxes; n_cam += len(boxes)
        if n_cam: print(f'  {cam_dir.name}: {n_cam} annotations')
    n_total = sum(len(v) for v in gt.values())
    cls_cnt = {}
    for boxes in gt.values():
        for b in boxes: cls_cnt[b['cls']] = cls_cnt.get(b['cls'],0)+1
    print(f'\n✓ GT: {len(gt)} images  {n_total} annotations')
    for c,n in sorted(cls_cnt.items()): print(f'  {c:15}: {n}')
    return gt

gt = load_gt(GT_ANN_ROOT, IMAGE_ROOT, GT_CLASSES, STRIP_HEIGHT)


## Cell 5 — Load crop run results

Reads all `results.csv` files from each crop run.
**Auto-detects pipeline names** from CSV column names — no manual config needed.

Also reads `run_config.json` to display the preprocessing parameters used,
so you know exactly what settings produced each set of results.

In [ ]:
def detect_pipelines_from_csv(csv_path):
    """Auto-detect pipeline names from CSV column names."""
    with open(csv_path,newline='') as f:
        fields = csv.DictReader(f).fieldnames or []
    # Pipeline columns follow pattern: {pipe_name}__binary_label or {pipe_name}__pollinator_type
    pipes = set()
    for field in fields:
        if '__binary_label' in field or '__pollinator_type' in field:
            pipes.add(field.split('__')[0])
    return sorted(pipes)

def load_crop_run(run_path, gt_image_paths):
    run_path = Path(run_path); gt_set = {str(p) for p in gt_image_paths}
    cfg_file = run_path/'run_config.json'
    run_cfg  = _json.loads(cfg_file.read_text()) if cfg_file.exists() else {}
    rows = []
    pipe_names = []
    for csv_path in sorted(run_path.rglob('results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(IMAGE_ROOT)/cam_name
        if not pipe_names:
            pipe_names = detect_pipelines_from_csv(csv_path)
        with open(csv_path,newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name','')
                img_p    = str(img_dir/img_name)
                if img_p not in gt_set:
                    stem = Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand = str(img_dir/(stem+ext))
                        if cand in gt_set: img_p=cand; break
                row['_img_path'] = img_p
                rows.append(row)
    matched = [r for r in rows if r['_img_path'] in gt_set]
    print(f'  Pipelines detected: {pipe_names}')
    print(f'  Rows: {len(rows)} total  {len(matched)} from GT images')
    # Show run config summary
    if run_cfg:
        pre = run_cfg.get('preprocess',{})
        print(f'  Config: large_motion={pre.get("enable_large_motion","?")}  '
              f'darker_threshold={pre.get("darker_threshold","?")}  '
              f'rolling_window={pre.get("rolling_window","?")}')
    return matched, pipe_names, run_cfg

print('Loading crop runs...')
crop_run_data = {}
for run_name, run_path in CROP_RUNS.items():
    print(f'\n  {run_name}:')
    rows, pipe_names, run_cfg = load_crop_run(run_path, gt.keys())
    crop_run_data[run_name] = {'rows':rows,'pipes':pipe_names,'config':run_cfg}
print('\n✓ Crop runs loaded.')


## Cell 6 — Matching + metrics functions

Defines the evaluation logic. **Do not edit.**

**`center_match`**: matches predicted bboxes to GT bboxes using center-point inclusion.
A prediction is a TP if its center falls inside a GT box, or a GT center falls inside the prediction.

**`evaluate_one_pipeline`**: for one pipeline, computes:
- **TP** — detected + correct class
- **FP** — detected but no matching GT (false alarm)
- **FN** — in GT but not detected (missed insect)
- **bg_rejected** — candidate crops the pipeline classified as background
  (these are FP rejections if there was a real insect there, or correct rejections if not)

In [ ]:
def bbox_overlap_ratio(p, g):
    """Overlap area / GT area — how much of GT is covered by prediction."""
    ix1 = max(p['x1'], g['x1']); iy1 = max(p['y1'], g['y1'])
    ix2 = min(p['x2'], g['x2']); iy2 = min(p['y2'], g['y2'])
    iw = max(0, ix2-ix1); ih = max(0, iy2-iy1)
    inter = iw * ih
    gt_area = max(1, (g['x2']-g['x1']) * (g['y2']-g['y1']))
    return inter / gt_area

def center_match(pred_boxes, gt_boxes, overlap_thresh=0.20):
    """
    Match predictions to GT using three criteria (any one is sufficient):
    1. GT center falls inside pred bbox
    2. Pred center falls inside GT bbox
    3. Overlap / GT area >= overlap_thresh (default 20%)
    """
    def inside(px, py, x1, y1, x2, y2):
        return x1 <= px <= x2 and y1 <= py <= y2

    mp=set(); mg=set(); pairs=[]
    for gi, g in enumerate(gt_boxes):
        gcx = (g['x1']+g['x2'])/2; gcy = (g['y1']+g['y2'])/2
        for pi, p in enumerate(pred_boxes):
            if pi in mp: continue
            pcx = (p['x1']+p['x2'])/2; pcy = (p['y1']+p['y2'])/2
            matched = (
                inside(gcx, gcy, p['x1'], p['y1'], p['x2'], p['y2']) or  # GT center in pred
                inside(pcx, pcy, g['x1'], g['y1'], g['x2'], g['y2']) or  # Pred center in GT
                bbox_overlap_ratio(p, g) >= overlap_thresh                # 20% GT coverage
            )
            if matched:
                pairs.append((pi, gi)); mp.add(pi); mg.add(gi); break
    return pairs, [i for i in range(len(pred_boxes)) if i not in mp], \
                  [i for i in range(len(gt_boxes)) if i not in mg]


## Cell 7 — Run all evaluations  ← **main evaluation cell**

Evaluates every pipeline from every run against GT.
For each crop run, automatically processes all detected pipelines.

Prints results per pipeline as it goes:
```
run_01/two_stage
  P=0.82  R=0.71  F1=0.76  TP=45  FP=10  FN=18  bg_rejected=23
```

In [ ]:
all_results = {}  # label -> metrics

# ── Crop runs ─────────────────────────────────────────────────────
for run_name, run_data in crop_run_data.items():
    rows      = run_data['rows']
    pipe_names= run_data['pipes']
    run_cfg   = run_data['config']
    pre       = run_cfg.get('preprocess',{})
    print(f'\n{"═"*60}')
    print(f'Crop run: {run_name}')
    print(f'  large_motion={pre.get("enable_large_motion","?")}  '
          f'darker_threshold={pre.get("darker_threshold","?")}')
    print(f'{"═"*60}')

    for pipe_name in pipe_names:
        p    = pipe_name + '__'
        label= f'{run_name}/{pipe_name}'

        # Detect pipeline type from columns
        sample_row = next((r for r in rows if r.get('pollinator_detected')=='yes'),{})
        is_two_stage = bool(sample_row.get(p+'binary_label',''))

        if is_two_stage:
            def get_cls(r, p=p):
                return r.get(p+'pollinator_type','') if r.get(p+'binary_label')=='insect' else ''
            def get_conf(r, p=p):
                try: return float(r.get(p+'group_conf') or r.get(p+'binary_conf') or 0)
                except: return 0.0
        else:
            def get_cls(r, p=p):
                pt=r.get(p+'pollinator_type',''); return '' if pt in ('background','') else pt
            def get_conf(r, p=p):
                try: return float(r.get(p+'group_conf') or 0)
                except: return 0.0

        all_results[label] = evaluate_one_pipeline(
            rows, gt, GT_CLASSES, get_cls, get_conf, label)

# ── YOLO runs ─────────────────────────────────────────────────────
for run_name, run_path in YOLO_RUNS.items():
    print(f'\n{"═"*60}\nYOLO run: {run_name}\n{"═"*60}')
    yolo_rows = []
    gt_set    = {str(p) for p in gt.keys()}
    for csv_path in sorted(Path(run_path).rglob('yolo_results.csv')):
        cam_name = csv_path.parent.name
        img_dir  = Path(IMAGE_ROOT)/cam_name
        with open(csv_path,newline='') as f:
            for row in csv.DictReader(f):
                img_name = row.get('image_name','')
                img_p    = str(img_dir/img_name)
                if img_p not in gt_set:
                    stem=Path(img_name).stem
                    for ext in ('.JPG','.jpg','.jpeg'):
                        cand=str(img_dir/(stem+ext))
                        if cand in gt_set: img_p=cand; break
                row['_img_path']=img_p
                row['pollinator_detected']='yes'
                row['bbox_x']=row.get('bbox_x',0)
                row['bbox_y']=row.get('bbox_y',0)
                row['bbox_w']=row.get('bbox_w',0)
                row['bbox_h']=row.get('bbox_h',0)
                yolo_rows.append(row)
    matched = [r for r in yolo_rows if r['_img_path'] in gt_set]
    print(f'  {len(matched)} detections from GT images')
    label = run_name
    all_results[label] = evaluate_one_pipeline(
        matched, gt, GT_CLASSES,
        lambda r: r.get('class_name',''),
        lambda r: float(r.get('confidence') or 0),
        label)

print(f'\n✓ Evaluated {len(all_results)} pipeline(s).')


## Cell 8 — Comparison table + save

Prints the full comparison table with all pipelines side by side.
Saves results to `evaluation/`:
- `summary.json` — all metrics in machine-readable format
- `eval_results.csv` — every crop labelled as tp/fp/fn/rejected_as_bg

The `eval_results.csv` is useful for deeper analysis —
you can filter it to see exactly which images/crops are causing FPs or FNs.

In [ ]:
# ── Master comparison table ──────────────────────────────────────
print('\n' + '═'*80)
print(f'{"Label":40}  {"P":>7}  {"R":>7}  {"F1":>7}  '
      f'{"TP":>5}  {"FP":>5}  {"FN":>5}  {"BG_rej":>7}')
print('═'*80)
for label, r in all_results.items():
    print(f'{label:40}  {r["precision"]:>7.3f}  {r["recall"]:>7.3f}  '
          f'{r["f1"]:>7.3f}  {r["tp"]:>5}  {r["fp"]:>5}  {r["fn"]:>5}  '
          f'{r["n_bg_rejected"]:>7}')

# ── Save eval_results.csv ────────────────────────────────────────
all_rows = [row for r in all_results.values() for row in r['rows']]
with open(EVAL_DIR/'eval_results.csv','w',newline='') as fh:
    w = csv.DictWriter(fh,
        fieldnames=['pipeline','img','match','pred_cls','gt_cls','conf','correct_cls'])
    w.writeheader(); w.writerows(all_rows)

# ── Save summary.json ────────────────────────────────────────────
def _ser(o):
    if isinstance(o,dict): return {str(k):_ser(v) for k,v in o.items()}
    if isinstance(o,(list,tuple)): return [_ser(x) for x in o]
    if isinstance(o,(np.integer,np.floating)): return o.item()
    return o
summary = {label:{k:v for k,v in r.items() if k!='rows'}
           for label,r in all_results.items()}
(EVAL_DIR/'summary.json').write_text(_json.dumps(_ser(summary),indent=2))
print(f'\n✓ Saved to {EVAL_DIR}')


## Cell 9 — PR curves

Plots precision-recall curves for all pipelines on one chart.
Each point on the curve corresponds to a different confidence threshold.

A curve closer to the top-right corner is better.
Compare curves to see which pipeline has the best precision-recall trade-off
for your use case (high recall = fewer missed insects, high precision = fewer false alarms).

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))
colors  = plt.cm.tab10.colors
linestyles = ['-','--','-.',':']

for i,(label,r) in enumerate(all_results.items()):
    dets = sorted(
        [(row['conf'],row['match']=='tp')
         for row in r['rows'] if row['match'] in ('tp','fp')],
        key=lambda x:-x[0])
    if not dets: continue
    n_gt=r['tp']+r['fn']; ps=[]; rs=[]; tp_=fp_=0
    for conf,is_tp in dets:
        if is_tp: tp_+=1
        else: fp_+=1
        ps.append(tp_/max(1,tp_+fp_)); rs.append(tp_/max(1,n_gt))
    ax.plot(rs,ps,
            label=f'{label} (F1={r["f1"]:.3f})',
            color=colors[i%len(colors)],
            ls=linestyles[i%len(linestyles)], lw=2)

ax.set_xlabel('Recall',fontsize=12); ax.set_ylabel('Precision',fontsize=12)
ax.set_title('Precision-Recall — all pipelines',fontsize=13)
ax.legend(fontsize=8,loc='upper right'); ax.grid(True,alpha=0.4)
ax.set_xlim(0,1); ax.set_ylim(0,1); plt.tight_layout()
plt.savefig(EVAL_DIR/'pr_curves.png',dpi=150)
print(f'✓ PR curves saved: {EVAL_DIR}/pr_curves.png')
plt.show()


## Cell 10 — Visualize pipelines on original images

Draws GT + all pipeline bboxes on original images and saves to `evaluation/viz/`.
Works on both **Colab** (images in Drive) and **locally** (images on disk).
Images are read from `IMAGE_ROOT` which is already set correctly in Cell 1.

Only generates images where at least one pipeline detected something or GT has an annotation.

Colours:
- 🟢 GT (ground truth)
- 🔵 two_stage
- 🟠 five_class_eff
- 🟣 five_class_ins
- 🔴 yolo

After this cell, run **Cell 11** to browse the images interactively.


In [ ]:
import cv2, numpy as np, csv
from pathlib import Path

# ── Colours per pipeline (BGR) ────────────────────────────────────
PIPELINE_COLORS = {
    'GT':              (0,   200,   0),   # green
    'two_stage':       (255, 100,   0),   # blue
    'five_class_eff':  (0,   140, 255),   # orange
    'five_class_ins':  (180,   0, 255),   # purple
    'yolo':            (0,    0, 255),    # red
}
DEFAULT_COLOR = (128, 128, 128)

FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.45
THICKNESS  = 2
LABEL_PAD  = 3

VIZ_DIR = EVAL_DIR / 'viz'
VIZ_DIR.mkdir(exist_ok=True)

def draw_bbox(img, x1, y1, x2, y2, label, color, offset_y=0):
    cv2.rectangle(img, (int(x1),int(y1)), (int(x2),int(y2)), color, THICKNESS)
    txt_y = max(int(y1) - 6 + offset_y, 12)
    (tw, th), _ = cv2.getTextSize(label, FONT, FONT_SCALE, 1)
    cv2.rectangle(img, (int(x1), txt_y-th-LABEL_PAD),
                       (int(x1)+tw+LABEL_PAD*2, txt_y+LABEL_PAD), color, -1)
    cv2.putText(img, label, (int(x1)+LABEL_PAD, txt_y),
                FONT, FONT_SCALE, (255,255,255), 1)

# ── Build index: image_path -> list of (pipeline, bbox, label) ───
print('Building image index...')
viz_index = {}  # rel_path -> list of (pipeline_label, x1,y1,x2,y2, color)

# From crop runs
for run_name, run_data in crop_run_data.items():
    for row in run_data['rows']:
        if row.get('pollinator_detected') != 'yes': continue
        try:
            x=int(row['bbox_x']); y=int(row['bbox_y'])
            w=int(row['bbox_w']); h=int(row['bbox_h'])
        except (ValueError,KeyError): continue
        img_p = row.get('image_path') or row.get('_img_path','')
        if not img_p: continue

        pipe_names = run_data['pipes']
        for pipe_name in pipe_names:
            p = pipe_name + '__'
            # Get predicted class for this pipeline
            if row.get(p+'binary_label') == 'insect':
                cls = row.get(p+'pollinator_type','insect') or 'insect'
                conf = row.get(p+'group_conf','')
            elif row.get(p+'binary_label') == 'background':
                cls = 'background'
                conf = row.get(p+'binary_conf','')
            elif row.get(p+'pollinator_type'):
                cls = row.get(p+'pollinator_type','')
                conf = row.get(p+'group_conf','')
                if cls == 'background': pass
            else:
                continue
            try: conf_str = f'{float(conf):.2f}' if conf else ''
            except: conf_str = ''
            label = f'{run_name}/{pipe_name}:{cls}' + (f'({conf_str})' if conf_str else '')
            color = PIPELINE_COLORS.get(pipe_name, DEFAULT_COLOR)
            viz_index.setdefault(img_p, []).append((label, x, y, x+w, y+h, color))

# From YOLO runs
for run_name, run_path in YOLO_RUNS.items():
    for csv_path in sorted(Path(run_path).rglob('yolo_results.csv')):
        cam_name = csv_path.parent.name
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                try:
                    x=int(row['bbox_x']); y=int(row['bbox_y'])
                    w=int(row['bbox_w']); h=int(row['bbox_h'])
                except (ValueError,KeyError): continue
                img_name = row.get('image_name','')
                # Build relative path same way as infer_cropbased
                rel_p = str(Path(cam_name) / img_name)
                cls = row.get('class_name',''); conf = row.get('confidence','')
                try: conf_str = f'{float(conf):.2f}' if conf else ''
                except: conf_str = ''
                label = f'{run_name}:{cls}({conf_str})'
                color = PIPELINE_COLORS.get('yolo', DEFAULT_COLOR)
                viz_index.setdefault(rel_p, []).append((label, x, y, x+w, y+h, color))

# Add GT boxes
for img_p, gt_boxes in gt.items():
    # Convert absolute path to relative for matching
    try: rel_p = str(Path(img_p).relative_to(IMAGE_ROOT.parent))
    except ValueError: rel_p = str(Path(img_p).name)
    for b in gt_boxes:
        label = f'GT:{b["cls"]}'
        viz_index.setdefault(rel_p, []).append(
            (label, b['x1'],b['y1'],b['x2'],b['y2'], PIPELINE_COLORS['GT']))

print(f'  {len(viz_index)} images to visualize')

# ── Generate viz for every image that has any annotation ─────────
# Only generate for images that have at least one detection or GT annotation
detected_paths = set()   # images with at least one pipeline detection
for run_name, run_data in crop_run_data.items():
    for row in run_data['rows']:
        if row.get('pollinator_detected') != 'yes': continue
        ip = row.get('image_path') or ''
        if ip: detected_paths.add(ip)

# Also include images that have GT annotations
gt_paths = set()
for img_p in gt.keys():
    try: rel_p = str(Path(img_p).relative_to(IMAGE_ROOT.parent))
    except ValueError: rel_p = Path(img_p).name
    gt_paths.add(rel_p)

# Also include images with YOLO detections
yolo_paths = set()
for run_name, run_path in YOLO_RUNS.items():
    for csv_path in sorted(Path(run_path).rglob('yolo_results.csv')):
        cam_name = csv_path.parent.name
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                yolo_paths.add(str(Path(cam_name)/row.get('image_name','')))

all_image_paths = detected_paths | gt_paths | yolo_paths

print(f'  {len(all_image_paths)} total inferred images')
print(f'Generating visualizations...')

n_saved = 0
for rel_p in sorted(all_image_paths):
    # Find actual image file
    full_p = IMAGE_ROOT.parent / rel_p
    if not full_p.exists():
        # try common extensions
        for ext in ('.JPG','.jpg','.jpeg'):
            cand = full_p.with_suffix(ext)
            if cand.exists(): full_p=cand; break
    if not full_p.exists(): continue

    img = cv2.imread(str(full_p))
    if img is None: continue

    # Strip OSD bar
    if STRIP_HEIGHT > 0 and img.shape[0] > STRIP_HEIGHT:
        img = img[:img.shape[0]-STRIP_HEIGHT, :]

    # Draw all boxes for this image
    boxes = viz_index.get(rel_p, [])
    # Sort: GT first, then pipelines
    gt_boxes_  = [(l,x1,y1,x2,y2,c) for l,x1,y1,x2,y2,c in boxes if l.startswith('GT')]
    pipe_boxes  = [(l,x1,y1,x2,y2,c) for l,x1,y1,x2,y2,c in boxes if not l.startswith('GT')]

    for i,(label,x1,y1,x2,y2,color) in enumerate(gt_boxes_):
        draw_bbox(img, x1,y1,x2,y2, label, color, offset_y=0)
    for i,(label,x1,y1,x2,y2,color) in enumerate(pipe_boxes):
        draw_bbox(img, x1,y1,x2,y2, label, color, offset_y=i*18)

    # Add legend top-left
    ly = 20
    for pipe_name, color in PIPELINE_COLORS.items():
        if pipe_name == 'GT': continue
        cv2.rectangle(img, (8, ly-10), (22, ly+2), color, -1)
        cv2.putText(img, pipe_name, (26, ly), FONT, 0.4, color, 1)
        ly += 18

    # Save with camera-folder prefix to avoid name clashes
    parts     = Path(rel_p).parts
    cam_name  = parts[0] if len(parts) > 1 else 'unknown'
    img_stem  = Path(rel_p).stem
    out_name  = f'{cam_name}__{img_stem}.jpg'
    out_path  = VIZ_DIR / out_name

    cv2.imwrite(str(out_path), img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    n_saved += 1
    if n_saved % 100 == 0:
        print(f'  {n_saved}/{len(all_image_paths)} saved...', flush=True)

print(f'\n✓ {n_saved} visualization images saved to:')
print(f'  {VIZ_DIR}')
print(f'\nTip: open the viz folder and browse images with any image viewer.')
print(f'  macOS: open in Finder, press Space to flip through')
print(f'  Windows: open folder, use arrow keys in Photos app')
print(f'  Linux: use eog, feh, or any image viewer')


## Cell 11 — Browse visualizations

Interactive slider to flip through visualization images.
Works on both Colab and local Jupyter.

Use the slider or type an image number to navigate.


In [ ]:
from IPython.display import display, Image as IPImage
import ipywidgets as widgets
from pathlib import Path

viz_images = sorted(Path(EVAL_DIR/'viz').glob('*.jpg'))
print(f'Found {len(viz_images)} visualization images')

if not viz_images:
    print('No images yet — run Cell 10 first.')
else:
    def show_image(i):
        p = viz_images[i]
        print(f'[{i+1}/{len(viz_images)}] {p.name}')
        display(IPImage(str(p), width=1200))

    slider = widgets.IntSlider(
        value=0, min=0, max=len(viz_images)-1, step=1,
        description='Image:',
        layout=widgets.Layout(width='80%')
    )
    widgets.interact(show_image, i=slider)
